In [2]:
import pandas as pd
import numpy as np

In [ ]:
# Reload datasets without treating first row as header
df1 = pd.read_csv("fin_trans.tsv", sep="\t", header=None, low_memory=False)
df2 = pd.read_csv("fin_account.tsv", sep="\t", header=None, low_memory=False)
df3 = pd.read_csv("fin_order.tsv", sep="\t", header=None, low_memory=False)

print("===== DATASET SHAPES =====")
print("Transaction:", df1.shape)
print("Account:", df2.shape)
print("Order:", df3.shape)

print("\n===== TRANSACTION SAMPLE =====")
print(df1.head(3))

print("\n===== ACCOUNT SAMPLE =====")
print(df2.head(3))

print("\n===== ORDER SAMPLE =====")
print(df3.head(3))

===== DATASET SHAPES =====
Transaction: (1056320, 10)
Account: (4500, 4)
Order: (6471, 6)

===== TRANSACTION SAMPLE =====
   0  1           2      3        4  5    6   7    8           9
0  1  1  2015-03-24  100.0   100.00  C  CIC      NaN         NaN
1  5  1  2015-04-13  367.9   467.90  C  COB       AB  41403269.0
2  6  1  2015-05-13  367.9  2097.72  C  COB       AB  41403269.0

===== ACCOUNT SAMPLE =====
   0   1           2  3
0  1  18  2015-03-24  M
1  2   1  2013-02-26  M
2  3   5  2017-07-07  M

===== ORDER SAMPLE =====
       0  1   2         3       4   5
0  29401  1  YZ  87144583  245.20  HH
1  29402  2  ST  89597016  337.27  LO
2  29403  2  QR  13943797  726.60  HH


## Assign Proper Column Names

In [6]:
# Transaction Dataset
df1.columns = [
    "trans_id",
    "account_id",
    "trans_date",
    "amount",
    "balance",
    "trans_type",
    "operation",
    "category",
    "other_bank_id",
    "other_account_id"
]

# Account Dataset
df2.columns = [
    "account_id",
    "district_id",
    "create_date",
    "frequency"
]

# Order Dataset
df3.columns = [
    "order_id",
    "account_id",
    "bank_to",
    "account_to",
    "amount",
    "category"
]

# Convert dates
df1["trans_date"] = pd.to_datetime(df1["trans_date"])
df2["create_date"] = pd.to_datetime(df2["create_date"])

# Check results
print("===== TRANSACTIONS =====")
print(df1.head(3))

print("\n===== ACCOUNTS =====")
print(df2.head(3))

print("\n===== ORDERS =====")
print(df3.head(3))

===== TRANSACTIONS =====
   trans_id  account_id trans_date  amount  balance trans_type operation  \
0         1           1 2015-03-24   100.0   100.00          C       CIC   
1         5           1 2015-04-13   367.9   467.90          C       COB   
2         6           1 2015-05-13   367.9  2097.72          C       COB   

  category other_bank_id  other_account_id  
0                    NaN               NaN  
1                     AB        41403269.0  
2                     AB        41403269.0  

===== ACCOUNTS =====
   account_id  district_id create_date frequency
0           1           18  2015-03-24         M
1           2            1  2013-02-26         M
2           3            5  2017-07-07         M

===== ORDERS =====
   order_id  account_id bank_to  account_to  amount category
0     29401           1      YZ    87144583  245.20       HH
1     29402           2      ST    89597016  337.27       LO
2     29403           2      QR    13943797  726.60       HH


## Full Dataset Health Check

In [7]:
datasets = {
    "TRANSACTIONS": df1,
    "ACCOUNTS": df2,
    "ORDERS": df3
}

for name, df in datasets.items():

    print(f"\n{'='*15} {name} {'='*15}")

    print("\nDataset Shape:")
    print(df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    print("\nData Types:")
    print(df.dtypes)

    print("\nUnique Accounts:")
    print(df["account_id"].nunique())


print("\n========== TRANSACTION ANALYSIS ==========")

print("\nTransaction Date Range:")
print("Start:", df1["trans_date"].min())
print("End:", df1["trans_date"].max())

print("\nTransaction Types:")
print(df1["trans_type"].value_counts(dropna=False))

print("\nNegative Amounts:")
print((df1["amount"] < 0).sum())

print("\nZero Amounts:")
print((df1["amount"] == 0).sum())

print("\nNegative Balances:")
print((df1["balance"] < 0).sum())


print("\n========== ACCOUNT INTEGRITY ==========")

print("Transaction Accounts Missing in Account Table:")
print(
    (~df1["account_id"].isin(df2["account_id"])).sum()
)

print("\nOrder Accounts Missing in Account Table:")
print(
    (~df3["account_id"].isin(df2["account_id"])).sum()
)


=============== TRANSACTIONS ===============

Dataset Shape:
(1056320, 10)

Missing Values:
trans_id                 0
account_id               0
trans_date               0
amount                   0
balance                  0
trans_type               0
operation                0
category                10
other_bank_id       760931
other_account_id    760931
dtype: int64

Duplicate Rows:
0

Data Types:
trans_id                     int64
account_id                   int64
trans_date          datetime64[ns]
amount                     float64
balance                    float64
trans_type                  object
operation                   object
category                    object
other_bank_id               object
other_account_id           float64
dtype: object

Unique Accounts:
4500

=============== ACCOUNTS ===============

Dataset Shape:
(4500, 4)

Missing Values:
account_id     0
district_id    0
create_date    0
frequency      0
dtype: int64

Duplicate Rows:
0

Data Types:
account

## Financial Transaction Validation

In [8]:
print("===== AMOUNT BY TRANSACTION TYPE =====")

print(
    df1.groupby("trans_type")["amount"].agg(
        ["count", "min", "max", "mean", "sum"]
    )
)

print("\n===== AMOUNT SIGN DISTRIBUTION =====")

df1["amount_sign"] = np.select(
    [
        df1["amount"] < 0,
        df1["amount"] == 0,
        df1["amount"] > 0
    ],
    ["Negative", "Zero", "Positive"],
    default="Unknown"
)

print(
    pd.crosstab(
        df1["trans_type"],
        df1["amount_sign"]
    )
)

print("\n===== TRANSACTION CATEGORY =====")
print(df1["category"].value_counts(dropna=False).head(20))

print("\n===== OPERATION TYPES =====")
print(df1["operation"].value_counts(dropna=False))

print("\n===== UNIQUE TRANSACTION IDs =====")
print(df1["trans_id"].nunique())

print("\n===== AMOUNT SUMMARY =====")
print(df1["amount"].describe())

===== AMOUNT BY TRANSACTION TYPE =====
             count     min     max         mean           sum
trans_type                                                   
C           405083     0.0  7481.2   796.743488  3.227472e+08
D           634571 -8740.0     0.0  -444.665353 -2.821717e+08
P            16666 -2499.7  -200.1 -1251.672615 -2.086038e+07

===== AMOUNT SIGN DISTRIBUTION =====
amount_sign  Negative  Positive  Zero
trans_type                           
C                   0    405079     4
D              634561         0    10
P               16666         0     0

===== TRANSACTION CATEGORY =====
category
       535304
IC     183114
ST     155832
HH     118065
PE      30338
IN      18500
LO      13580
IO       1577
NaN        10
Name: count, dtype: int64

===== OPERATION TYPES =====
operation
WIC    434918
ROB    208283
       183114
CIC    156743
COB     65226
CCW      8036
Name: count, dtype: int64

===== UNIQUE TRANSACTION IDs =====
1056320

===== AMOUNT SUMMARY =====
count  

## Account History & Forecasting Readiness

In [9]:
# Account-level transaction history

account_history = df1.groupby("account_id").agg(
    total_transactions=("trans_id", "count"),
    first_transaction=("trans_date", "min"),
    last_transaction=("trans_date", "max"),
    total_cash_flow=("amount", "sum")
)

# Calculate history duration
account_history["history_days"] = (
    account_history["last_transaction"]
    - account_history["first_transaction"]
).dt.days

account_history["history_months"] = (
    account_history["history_days"] / 30.44
).round(1)

print("===== ACCOUNT HISTORY SUMMARY =====")
print(account_history.describe())

print("\n===== FORECASTING READINESS =====")

print("Total Accounts:", len(account_history))

print(
    "Accounts with 6+ Months:",
    (account_history["history_months"] >= 6).sum()
)

print(
    "Accounts with 12+ Months:",
    (account_history["history_months"] >= 12).sum()
)

print(
    "Accounts with 24+ Months:",
    (account_history["history_months"] >= 24).sum()
)

print(
    "Accounts with 36+ Months:",
    (account_history["history_months"] >= 36).sum()
)

print("\n===== TRANSACTIONS PER ACCOUNT =====")
print(account_history["total_transactions"].describe())

print("\n===== HISTORY MONTHS =====")
print(account_history["history_months"].describe())

===== ACCOUNT HISTORY SUMMARY =====
       total_transactions           first_transaction  \
count         4500.000000                        4500   
mean           234.737778  2015-08-08 01:38:52.800000   
min              9.000000         2013-01-01 00:00:00   
25%            133.000000         2013-12-27 00:00:00   
50%            208.000000         2016-01-02 00:00:00   
75%            330.000000         2016-11-01 00:00:00   
max            675.000000         2017-12-29 00:00:00   
std            126.849443                         NaN   

                    last_transaction  total_cash_flow  history_days  \
count                           4500      4500.000000   4500.000000   
mean   2018-12-28 09:37:16.800000256      4381.139749   1238.332222   
min              2016-08-25 00:00:00     -2582.100000    330.000000   
25%              2018-12-31 00:00:00      2307.510000    786.000000   
50%              2018-12-31 00:00:00      3849.485000   1093.000000   
75%              2018-12

## Monthly Activity Validation

In [10]:
# Create monthly period
df1["year_month"] = df1["trans_date"].dt.to_period("M")

# Number of active months per account
monthly_activity = (
    df1.groupby("account_id")["year_month"]
    .nunique()
    .reset_index(name="active_months")
)

# Calendar months between first and last transaction
account_history["calendar_months"] = (
    (
        account_history["last_transaction"].dt.year
        - account_history["first_transaction"].dt.year
    ) * 12
    +
    (
        account_history["last_transaction"].dt.month
        - account_history["first_transaction"].dt.month
    )
    + 1
)

# Combine results
monthly_activity = monthly_activity.merge(
    account_history[["calendar_months"]],
    left_on="account_id",
    right_index=True
)

# Calculate inactive months
monthly_activity["inactive_months"] = (
    monthly_activity["calendar_months"]
    - monthly_activity["active_months"]
)

# Activity ratio
monthly_activity["activity_ratio"] = (
    monthly_activity["active_months"]
    / monthly_activity["calendar_months"]
)

print("===== MONTHLY ACTIVITY SUMMARY =====")
print(monthly_activity.describe())

print("\n===== ACCOUNT ACTIVITY LEVELS =====")

print(
    "Accounts with 90%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.90).sum()
)

print(
    "Accounts with 75%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.75).sum()
)

print(
    "Accounts with 50%+ Active Months:",
    (monthly_activity["activity_ratio"] >= 0.50).sum()
)

print("\n===== INACTIVE MONTHS =====")
print(monthly_activity["inactive_months"].describe())

print("\n===== ACCOUNT ELIGIBILITY =====")

eligible = monthly_activity[
    (monthly_activity["calendar_months"] >= 12)
    & (monthly_activity["activity_ratio"] >= 0.75)
]

print("Potential Eligible Accounts:", len(eligible))

===== MONTHLY ACTIVITY SUMMARY =====
         account_id  active_months  calendar_months  inactive_months  \
count   4500.000000    4500.000000      4500.000000      4500.000000   
mean    2786.067556      41.123778        41.183556         0.059778   
std     2313.811984      18.153406        18.131747         0.612335   
min        1.000000       9.000000        12.000000         0.000000   
25%     1182.750000      26.000000        26.000000         0.000000   
50%     2368.000000      36.000000        36.000000         0.000000   
75%     3552.250000      61.000000        61.000000         0.000000   
max    11382.000000      72.000000        72.000000        16.000000   

       activity_ratio  
count     4500.000000  
mean         0.998159  
std          0.018736  
min          0.500000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  

===== ACCOUNT ACTIVITY LEVELS =====
Accounts with 90%+ Active Months: 4479
Accounts with 75%+ Act

## Monthly Financial Aggregation

In [11]:
# Work with eligible accounts only
eligible_ids = eligible["account_id"]

df_monthly = df1[
    df1["account_id"].isin(eligible_ids)
].copy()

# Create financial features
df_monthly["income"] = df_monthly["amount"].clip(lower=0)

df_monthly["expense"] = (
    -df_monthly["amount"].clip(upper=0)
)

# Sort transactions
df_monthly = df_monthly.sort_values(
    ["account_id", "trans_date", "trans_id"]
)

# Aggregate monthly financial data
monthly_finance = (
    df_monthly.groupby(
        ["account_id", "year_month"]
    )
    .agg(
        income=("income", "sum"),
        expense=("expense", "sum"),
        transaction_count=("trans_id", "count"),
        month_end_balance=("balance", "last")
    )
    .reset_index()
)

# Net cash flow
monthly_finance["net_cash_flow"] = (
    monthly_finance["income"]
    - monthly_finance["expense"]
)

# Check results
print("===== MONTHLY FINANCIAL DATASET =====")
print(monthly_finance.head(10))

print("\n===== DATASET SHAPE =====")
print(monthly_finance.shape)

print("\n===== UNIQUE ACCOUNTS =====")
print(monthly_finance["account_id"].nunique())

print("\n===== MONTHLY FINANCIAL SUMMARY =====")
print(
    monthly_finance[
        ["income", "expense", "net_cash_flow", "month_end_balance"]
    ].describe()
)

print("\n===== MISSING VALUES =====")
print(monthly_finance.isnull().sum())

===== MONTHLY FINANCIAL DATASET =====
   account_id year_month   income  expense  transaction_count  \
0           1    2015-03   100.00     0.00                  1   
1           1    2015-04  1629.82     0.00                  3   
2           1    2015-05   585.80     0.00                  3   
3           1    2015-06   377.96    20.00                  3   
4           1    2015-07   378.79   530.00                  3   
5           1    2015-08   377.82   751.46                  4   
6           1    2015-09   377.05   316.66                  5   
7           1    2015-10   376.95   536.66                  5   
8           1    2015-11   376.42   523.66                  6   
9           1    2015-12   375.80   460.66                  6   

   month_end_balance  net_cash_flow  
0             100.00         100.00  
1            1729.82        1629.82  
2            2315.62         585.80  
3            2673.58         357.96  
4            2522.37        -151.21  
5            2150.

## Monthly Timeline Validation

In [12]:
# STEP 9: Monthly Timeline Validation

monthly_finance = monthly_finance.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Check actual calendar gaps
monthly_finance["previous_month"] = (
    monthly_finance.groupby("account_id")["year_month"]
    .shift(1)
)

monthly_finance["month_gap"] = (
    (monthly_finance["year_month"].dt.year -
     monthly_finance["previous_month"].dt.year) * 12
    +
    (monthly_finance["year_month"].dt.month -
     monthly_finance["previous_month"].dt.month)
)

# Gaps greater than 1 month
missing_month_gaps = monthly_finance[
    monthly_finance["month_gap"] > 1
].copy()

missing_month_gaps["missing_months"] = (
    missing_month_gaps["month_gap"] - 1
)

print("===== MONTHLY TIMELINE REPORT =====")

print("Total Monthly Records:", len(monthly_finance))

print(
    "Accounts with Missing Months:",
    missing_month_gaps["account_id"].nunique()
)

print(
    "Total Missing Calendar Months:",
    int(missing_month_gaps["missing_months"].sum())
)

print("\n===== MONTH GAP DISTRIBUTION =====")
print(monthly_finance["month_gap"].value_counts().sort_index())

print("\n===== ACCOUNTS WITH LARGEST GAPS =====")

print(
    missing_month_gaps[
        ["account_id", "year_month", "month_gap", "missing_months"]
    ]
    .sort_values("missing_months", ascending=False)
    .head(10)
)

===== MONTHLY TIMELINE REPORT =====
Total Monthly Records: 184903
Accounts with Missing Months: 4493
Total Missing Calendar Months: 108729635

===== MONTH GAP DISTRIBUTION =====
month_gap
1        180271
2           118
3            11
4             4
5             2
          ...  
24225        83
24226        83
24227        78
24228        62
24229        64
Name: count, Length: 68, dtype: int64

===== ACCOUNTS WITH LARGEST GAPS =====
        account_id year_month  month_gap  missing_months
46195         1171    2017-12      24229           24228
46653         1186    2017-12      24229           24228
30568          777    2017-12      24229           24228
113970        2918    2017-12      24229           24228
112936        2892    2017-12      24229           24228
29785          758    2017-12      24229           24228
158964        4332    2017-12      24229           24228
131784        3385    2017-12      24229           24228
5272           134    2017-12      24229     

## Corrected Code

In [13]:
# STEP 9: FIXED MONTHLY TIMELINE VALIDATION

monthly_finance = monthly_finance.sort_values(
    ["account_id", "year_month"]
).reset_index(drop=True)

# Convert PeriodIndex to numeric month index
monthly_finance["month_number"] = (
    monthly_finance["year_month"].dt.year * 12
    + monthly_finance["year_month"].dt.month
)

# Calculate actual month gaps
monthly_finance["month_gap"] = (
    monthly_finance.groupby("account_id")["month_number"]
    .diff()
)

# First month of each account has no previous month
monthly_finance["month_gap"] = (
    monthly_finance["month_gap"].fillna(0).astype(int)
)

# Identify missing months
missing_month_gaps = monthly_finance[
    monthly_finance["month_gap"] > 1
].copy()

missing_month_gaps["missing_months"] = (
    missing_month_gaps["month_gap"] - 1
)

print("===== CORRECTED MONTHLY TIMELINE REPORT =====")

print("Total Monthly Records:", len(monthly_finance))

print(
    "Accounts with Missing Months:",
    missing_month_gaps["account_id"].nunique()
)

print(
    "Total Missing Calendar Months:",
    missing_month_gaps["missing_months"].sum()
)

print("\n===== MONTH GAP DISTRIBUTION =====")
print(
    monthly_finance["month_gap"]
    .value_counts()
    .sort_index()
)

print("\n===== LARGEST GAPS =====")
print(
    missing_month_gaps[
        ["account_id", "year_month", "month_gap", "missing_months"]
    ]
    .sort_values("missing_months", ascending=False)
    .head(10)
)

===== CORRECTED MONTHLY TIMELINE REPORT =====
Total Monthly Records: 184903
Accounts with Missing Months: 96
Total Missing Calendar Months: 185

===== MONTH GAP DISTRIBUTION =====
month_gap
0      4493
1    180271
2       118
3        11
4         4
5         2
6         1
7         2
9         1
Name: count, dtype: int64

===== LARGEST GAPS =====
        account_id year_month  month_gap  missing_months
9297           236    2018-12          9               8
148523        3823    2017-03          7               6
77802         1974    2018-05          7               6
79687         2021    2018-07          6               5
62244         1589    2018-08          5               4
80064         2029    2016-09          5               4
89732         2300    2018-05          4               3
44946         1138    2018-07          4               3
31376          799    2016-08          4               3
17329          438    2018-11          4               3


## Create Complete Monthly Timeline